In [1]:
from openaq import OpenAQ
from dotenv import load_dotenv
import os
from pprint import pprint
import json
from tqdm import tqdm
import pandas as pd
from time import sleep
import logging

logging.basicConfig(filename='scrape_data.log', level=logging.ERROR, format='%(asctime)s:%(levelname)s:%(message)s')

load_dotenv(override=True)
openaq_api_key = os.getenv("OPENAQ_API_KEY")
openaq_client = OpenAQ(api_key=openaq_api_key)

In [2]:
def print_pretty(object):
    pprint(object, width=200)

In [3]:
from datetime import datetime, timedelta

def split_period(start, end):
    start = datetime.fromisoformat(start)
    end = datetime.fromisoformat(end)
    
    year_count = abs(end.year - start.year) + 1
    n = 12 * year_count

    delta = (end - start) / n
    intervals = []
    for i in range(n):
        begin = start + i * delta
        finish = begin + delta
        intervals.append((begin.isoformat(), finish.isoformat()))
    return intervals

In [ ]:
# lấy tất cả các trạm quan trắc ở Việt Nam
vn_locations = openaq_client.locations.list(limit=1000, countries_id=56)
print(len(vn_locations.results))

53


In [6]:
# lấy các trạm quan trắc ở Hà Nội
hn_locations = [loc for loc in vn_locations.results if loc.locality == "Hanoi"]

In [ ]:
try:
    for loc in hn_locations:
        loc_sensors = openaq_client.locations.sensors(loc.id)

        if loc_sensors.headers.x_ratelimit_remaining == 0:
            sleep(60)
            openaq_client.close()
            openaq_client = OpenAQ(api_key=openaq_api_key)

        for loc_sensor in loc_sensors.results:
            datetime_first = loc_sensor.datetime_first
            datetime_last = loc_sensor.datetime_last
            periods = split_period(datetime_first.utc, datetime_last.utc)
            for period in tqdm(periods, desc=f"{datetime_first.utc} - {datetime_last.utc}"):
                measurements = openaq_client.measurements.list(sensors_id=loc_sensor.id, datetime_from=period[0], datetime_to=period[1], limit=1000)
                if measurements.meta.found > 0:
                    df = pd.DataFrame(measurements.results)
                    df.to_csv(f"../csv_data/loc{loc.id}_sen{loc_sensor.id}.csv", index=False, encoding="utf-8", mode="a")
                
                if measurements.headers.x_ratelimit_remaining == 0:
                    sleep(60)
                    openaq_client.close()
                    openaq_client = OpenAQ(api_key=openaq_api_key)
                sleep(0.3)

except Exception as e:
    logging.error(f"Error processing location {loc.id}, sensor {loc_sensor.id}, period {period[0]} to {period[1]}: {e}")
    log = {"location_id": loc.id, "sensor_id": loc_sensor.id, "period_start": period[0], "period_end": period[1], "error": str(e)}
    pd.DataFrame([log]).to_csv("../csv_scrape_error/scrape_error.csv", index=False, encoding="utf-8", mode="a")
except KeyboardInterrupt as ke:
    logging.error("Scraping interrupted by user.")
    print("Scraping interrupted by user.")
    log = {"location_id": loc.id, "sensor_id": loc_sensor.id, "period_start": period[0], "period_end": period[1], "error": str(ke)}
    pd.DataFrame([log]).to_csv("../csv_scrape_error/scrape_error.csv", index=False, encoding="utf-8", mode="a")

2024-01-29T06:00:00Z - 2025-06-10T03:00:00Z:   8%|▊         | 2/24 [00:04<00:53,  2.42s/it]

Scraping interrupted by user.


In [ ]:
# import ast
# ast.literal_eval(dff.loc[0, "period"])

{'label': 'raw',
 'interval': '01:00:00',
 'datetime_from': {'utc': '2020-06-23T13:00:00Z',
  'local': '2020-06-23T20:00:00+07:00'},
 'datetime_to': {'utc': '2020-06-23T14:00:00Z',
  'local': '2020-06-23T21:00:00+07:00'}}